# test plotting

In [ ]:
import pandas as pd
from matplotlib.ticker import ScalarFormatter
import matplotlib.pyplot as plt
import seaborn as sns
import click
from scipy import stats

In [ ]:
X_VAR = "N_Per_Split"
Y_VAR = "Fraction"
X_LABEL = "Number of Reference Structures Available to Use"
Y_LABEL = "Fraction of Ligands Posed \n<2Å from Reference"
COLOR_VAR = "Score"
STYLE_VAR = "Split"
CI_LOWER = "CI_Lower"
CI_UPPER = "CI_Upper"
LARGE_FIG_SIZE = (12, 8)
SMALL_FIG_SIZE = (8, 6)
FONT_SIZES = {
    "xlabel": 24,
    "ylabel": 24,
    "ticks": 18,
    "legend_title": 24,
    "legend_text": 18,
}
ALPHA = 0.2

In [ ]:
ligand_similarity_csv = "/data1/choderaj/paynea/asap-datasets/full_cross_dock/chemical_similarity_data/combined_chemical_similarity_data.csv"

In [ ]:
df = pd.read_csv(ligand_similarity_csv)

# Filter based on most interesting
df = df[df["bitsize"].isin([2048]) | df["bitsize"].isna()]
df = df[df["radius"].isin([2, 5]) | df["radius"].isna()]

color_var = "Similarity_Metric"

# make a column that is the combination of the relevant variables
# TODO: this is bad and hard-coded but to change it I'd have to go all the way back the chemical_similarity_schema
df[color_var] = (
    df["Type"].astype(str)
    + "_"
    + df["Aligned"].astype(str)
    + "_"
    + df["radius"].astype(str)
    + "_"
    + df["bitsize"].astype(str)
)

In [ ]:
x_var = "Tanimoto"
df = df.sort_values(by=[color_var, x_var])

In [ ]:
# Create ECDF
plt.figure(figsize=LARGE_FIG_SIZE)
fig = sns.ecdfplot(df, x=x_var, hue=color_var, stat="proportion", linewidth=4)
plt.xlabel("Tanimoto Similarity", fontsize=FONT_SIZES["xlabel"], fontweight="bold")
plt.ylabel(
    "Fraction of Pairwise\n Ligand Similarities",
    fontsize=FONT_SIZES["ylabel"],
    fontweight="bold",
)

# tick text
plt.xticks(fontsize=FONT_SIZES["ticks"])
plt.yticks(fontsize=FONT_SIZES["ticks"])

# for legend text
plt.setp(fig.get_legend().get_texts(), fontsize=FONT_SIZES["legend_text"])
plt.setp(fig.get_legend().get_title(), fontsize=FONT_SIZES["legend_title"])

# Make Scaffold X TO Y HEATMAP

In [ ]:
x_to_y = pd.read_csv("/data1/choderaj/paynea/asap-datasets/full_cross_dock/analyzed_results/x_to_y_posit_combined_results.csv")

In [ ]:
raw_df = x_to_y.copy()

In [ ]:
ref = "Reference_Scaffold_ID_Subset_1"
query = "Query_Scaffold_ID_Subset_1"

In [ ]:
raw_df[query] = raw_df[query].astype(str).apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
raw_df["qint"] = raw_df[query].astype(float)
raw_df[ref] = raw_df[ref].astype(str).apply(lambda x: x.replace("[", "").replace("]", "") if "[" in x else x)
raw_df["rint"] = raw_df[ref].astype(float)

In [ ]:
groupby = ["Score", "Split"]

In [ ]:
heatmap_dfs = {" | ".join(name): group for name, group in raw_df.groupby(groupby)}

In [ ]:
heatmap_dfs.keys()

In [ ]:
heatmap_rmsd_random = raw_df[(raw_df["Score"] == "RMSD")&(raw_df["Split"] == "RandomSplit")]

In [ ]:
pivot_fraction = heatmap_rmsd_random.pivot(index='qint',
                                         columns='rint',
                                         values='Fraction')

In [ ]:
pivot_total = heatmap_rmsd_random.pivot(index='qint',
                                       columns='rint',
                                       values='Total')

In [ ]:
ref_counts = heatmap_rmsd_random.sort_values('rint').groupby(ref).head(1)[[ref, "Total"]].to_dict(orient="records")

In [ ]:
count_dict = {data[ref]: data["Total"] for data in ref_counts}

In [ ]:
ytick_labels = [f"$\\bf{cluster_id}$ ({total})" for cluster_id, total in count_dict.items()]
xtick_labels = [f"$\\bf{cluster_id}$\n({total})" for cluster_id, total in count_dict.items()]

## heatmap

In [ ]:
plt.figure(figsize=LARGE_FIG_SIZE)
heatmap = sns.heatmap(data=pivot_fraction,
                      xticklabels=xtick_labels,
                      yticklabels=ytick_labels,
                      annot=True,
                     cmap="coolwarm_r")
# Add colorbar label
heatmap.collections[0].colorbar.set_label('Fraction')

# Rotate axis labels for better readability
plt.xticks(rotation=0)
plt.yticks(rotation=0)

# Invert y-axis to put 0 at bottom
plt.gca().invert_yaxis()

# violin comparing refs to query

In [ ]:
ref_df = pd.DataFrame({"Role": "Reference", 
                       "Fraction": heatmap_rmsd_random.Fraction, 
                       "Scaffold": heatmap_rmsd_random.Reference_Scaffold_ID_Subset_1})

In [ ]:
query_df = pd.DataFrame({"Role": "Query", 
                       "Fraction": heatmap_rmsd_random.Fraction, 
                       "Scaffold": heatmap_rmsd_random.Query_Scaffold_ID_Subset_1})

In [ ]:
tidy_df = pd.concat([ref_df, query_df])
tidy_df['sint'] = tidy_df.Scaffold.astype(int)

In [ ]:
stats_df = tidy_df.groupby(['Scaffold', 'sint', 'Role'])["Fraction"].describe()

In [ ]:
std_df = stats_df['std'].reset_index()

In [ ]:
ref_stds = std_df[std_df["Role"] == "Reference"]["std"].tolist()

In [ ]:
query_stds = std_df[std_df["Role"] == "Query"]["std"].tolist()

In [ ]:
t_stat, p_value = stats.ttest_rel(ref_stds, query_stds)
print("T-statistic:", t_stat)
print("P-value:", p_value)

In [ ]:
sns.boxplot(std_df, y="std", x='Role')

In [ ]:
sns.scatterplot(x=ref_stds, y=query_stds)

# Get current axis limits
x_min, x_max = plt.xlim()
y_min, y_max = plt.ylim()

# Calculate y=x line endpoints
start = min(x_min, y_min)
end = max(x_max, y_max)

# Plot the y=x line
plt.plot([start, end], [start, end], color='red', linestyle='--', label='y=x')

# Add legend and show the plot
plt.legend()
plt.show()

In [ ]:
std_df.sort_values('sint', inplace=True)

In [ ]:
sns.catplot(data=std_df, x="Role", y="std", hue="Scaffold", kind="point", palette="cubehelix")

In [ ]:
sns.catplot(data=tidy_df, hue="Role", y="Fraction", x="sint", kind="box", palette="cubehelix")

In [ ]:
sns.boxplot(tidy_df, hue="Role", y="Fraction", x="sint")

In [ ]:
sns.violinplot(data=tidy_df, x="sint", y="Fraction", inner="point", hue="Role", split=True, gap=.1,)

In [ ]:
sns.scatterplot(tidy_df, hue="Role", y="Fraction", x="sint")

In [ ]:
tidy_df[(tidy_df["Role"]=="Reference")&(tidy_df["Scaffold"] == scaffold)]

In [ ]:
ref_df = heatmap_rmsd_random[heatmap_rmsd_random[ref] == '3']

In [ ]:
ref_df

In [ ]:
scaffold_dfs = [ ]
for scaffold in heatmap_rmsd_random[ref].unique():
    ref_df = heatmap_rmsd_random[heatmap_rmsd_random[ref] == scaffold]
    ref_half_df = pd.DataFrame({"Ref_Fraction": ref_df.Fraction.tolist(), "Partner": ref_df[query].to_list(), "Scaffold": scaffold})

    query_df = heatmap_rmsd_random[heatmap_rmsd_random[query] == scaffold]
    query_half_df = pd.DataFrame({"Query_Fraction": query_df.Fraction.tolist(), "Partner": query_df[ref].to_list(), "Scaffold": scaffold})

    # merge them
    scaff_df = ref_half_df.merge(query_half_df, on=['Partner', 'Scaffold'])
    scaffold_dfs.append(scaff_df)
by_scaff = pd.concat(scaffold_dfs)

In [ ]:
by_scaff['Scaffold'] = by_scaff.Scaffold.astype(int)

In [ ]:
by_scaff

## this is a good case for a paired t test

In [ ]:
t_stat, p_value = stats.ttest_rel(by_scaff['Ref_Fraction'], by_scaff['Query_Fraction'])
print("T-statistic:", t_stat)
print("P-value:", p_value)

In [ ]:
g = sns.FacetGrid(by_scaff, col="Scaffold", col_wrap=4)
g.map(sns.scatterplot, 'Ref_Fraction', 'Query_Fraction')

In [ ]:
spears = []
taus = []
pears = []
scaffs = []
for scaff in by_scaff.Scaffold.unique().tolist():
    scaffs.append(int(scaff))
    df = by_scaff[by_scaff.Scaffold == scaff]
    pears.append(df['Query_Fraction'].corr(df['Ref_Fraction'], method='pearson'))
    taus.append(df['Query_Fraction'].corr(df['Ref_Fraction'], method='kendall'))
    spears.append(df['Query_Fraction'].corr(df['Ref_Fraction'], method='spearman'))
corr_dfs = [pd.DataFrame({"Value": data, "Type": name, "Scaffold":scaffs}) 
            for name, data in zip(["Spearman", "Kendall", "Pearson"], [spears, taus, pears])]
corr_df = pd.concat(corr_dfs)

In [ ]:
sns.barplot(corr_df, x='Scaffold', y='Value', hue='Type')

In [ ]:
by_scaff

# How they vary with total

In [ ]:
heatmap_rmsd_random["Ref_Total"] = heatmap_rmsd_random.Reference_Scaffold_ID_Subset_1.apply(lambda x: count_dict[x])

In [ ]:
sns.scatterplot(heatmap_rmsd_random, x="Ref_Total", y="Fraction")

# Why differences between RMSD and DateSplit

In [ ]:
raw_df[raw_df[ref] == raw_df[query]].sort_values([query, ref])

In [ ]:
from harbor.analysis.cross_docking import Settings, Evaluator, FractionGood, Results

In [ ]:
all_sim = pd.read_csv("/data1/choderaj/paynea/asap-datasets/full_cross_dock/combined_docking_results/ALL_combined_results.csv")

In [ ]:
settings = Settings()
settings.n_bootstraps = 10
settings.query_scaffold_min_count = 70
settings.reference_scaffold_min_count = 70
settings.use_posit_scorer = True
settings.use_rmsd_scorer = False
settings.combine_core_and_chemical_splits = True
settings.n_per_split = 600
settings.update_n_per_split = False
settings.use_scaffold_split = True
settings.scaffold_split_option = 'x_to_y'
settings.use_random_split = True
settings.use_date_split = True

In [ ]:
settings

In [ ]:
evs = settings.create_evaluators(all_sim)

In [ ]:
summary_df = pd.DataFrame.from_records(
            [ev.get_records() for ev in evs]
        )

In [ ]:
summary_df

In [ ]:
randomsplit = evs[0].dataset_split

In [ ]:
datesplit = evs[1].dataset_split

In [ ]:
posefilter = evs[0].pose_selector

In [ ]:
testdf = posefilter.run(all_sim)

In [ ]:
testdf.nunique()

In [ ]:
randomized = randomsplit.run(testdf)[0]

In [ ]:
datified = datesplit.run(testdf)[0]

In [ ]:
randomized.nunique()

In [ ]:
datified.nunique()

In [ ]:
results = list(Results.calculate_results(all_sim, evs))

In [ ]:
results_df = Results.df_from_results(results)

In [ ]:
results_df

In [ ]:
all_sim.nunique()

In [ ]:
date_split_ev = evs[1]

In [ ]:
pose1 = date_split_ev.pose_selector.run(all_sim)

In [ ]:
pose1.nunique()

In [ ]:
ds = dataset_split.copy()
ds.n_per_split = 600
results_dfs = run(ds, pose1)[0]

In [ ]:
results_dfs.nunique()

In [ ]:
random_split_subset_df[~random_split_subset_df.index.isin(date_split_subset_df.index)].nunique()

In [ ]:
date_split_subset_df[~date_split_subset_df.index.isin(random_split_subset_df.index)].nunique()

# Test to Make sure Pose Selector isn't fucking things up

In [ ]:
all_sim.nunique()

In [ ]:
from harbor.analysis.cross_docking import PoseSelector

In [ ]:
groupby_for_pose_selector = PoseSelector(variable="Pose_ID", name="Default").groupby

In [ ]:
groupby_for_pose_selector

In [ ]:
nunique = all_sim.groupby(groupby_for_pose_selector).nunique()

In [ ]:
# Filter to keep only columns where at least one value is > 1
duplicates = nunique.loc[:, (nunique > 1).any()]

In [ ]:
duplicates